# Day 21: Elastic Net Regression

Learn regularized linear regression using Elastic Net. Includes scaling, training, evaluation, alpha and l1_ratio experiments, comparison with Linear Regression/Ridge/Lasso, coefficient inspection, and exercises.

Install if needed: `pip install pandas numpy matplotlib scikit-learn`.

## 1. Import libraries and load data
The California Housing dataset contains housing features and a numerical median house-value target. It may download on first use.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

data = fetch_california_housing(as_frame=True)
X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("Data shape:", X.shape)
display(X.head())
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import ElasticNet, LinearRegression, Ridge, Lasso

## 2. Train Elastic Net Regression
Elastic Net combines L1 (Lasso) and L2 (Ridge) regularization. Feature scaling is important. A Pipeline fits the scaler using training data only.

In [ ]:
model=Pipeline([
    ("scaler",StandardScaler()),
    ("elasticnet",ElasticNet(alpha=0.1,l1_ratio=0.5,max_iter=10000))
])
model.fit(X_train,y_train)
print("Model trained!")

## 3. Evaluate the model

In [ ]:
pred = model.predict(X_test)
print("MAE:", round(mean_absolute_error(y_test, pred), 3))
print("RMSE:", round(np.sqrt(mean_squared_error(y_test, pred)), 3))
print("R2:", round(r2_score(y_test, pred), 3))
display(pd.DataFrame({"Actual": y_test.to_numpy(), "Predicted": pred}).head(10))
plt.figure(figsize=(6,5))
plt.scatter(y_test, pred, alpha=0.25)
a,b=min(y_test.min(),pred.min()),max(y_test.max(),pred.max())
plt.plot([a,b],[a,b],"--")
plt.xlabel("Actual"); plt.ylabel("Predicted"); plt.title("Actual vs Predicted")
plt.show()

## 4. Understand alpha and l1_ratio
`alpha` controls the overall regularization strength. `l1_ratio=1` is Lasso-like; `l1_ratio=0` is Ridge-like; values between them mix both penalties.

In [ ]:
for ratio in [0.1,0.5,0.9]:
    trial=Pipeline([("scaler",StandardScaler()),
                    ("reg",ElasticNet(alpha=0.1,l1_ratio=ratio,max_iter=10000))])
    trial.fit(X_train,y_train); p=trial.predict(X_test)
    print(f"l1_ratio={ratio:.1f} | MAE={mean_absolute_error(y_test,p):.3f} | R2={r2_score(y_test,p):.3f}")

## 5. Compare Linear Regression, Ridge, Lasso, and Elastic Net

In [ ]:
models={
 "Linear Regression":LinearRegression(),
 "Ridge":Ridge(alpha=1.0),
 "Lasso":Lasso(alpha=0.1,max_iter=10000),
 "Elastic Net":ElasticNet(alpha=0.1,l1_ratio=0.5,max_iter=10000)
}
rows=[]
for name,est in models.items():
    pipe=Pipeline([("scaler",StandardScaler()),("model",est)])
    pipe.fit(X_train,y_train); p=pipe.predict(X_test)
    rows.append({"Model":name,"MAE":mean_absolute_error(y_test,p),
                 "RMSE":np.sqrt(mean_squared_error(y_test,p)),"R2":r2_score(y_test,p)})
display(pd.DataFrame(rows).sort_values("RMSE"))

## 6. Inspect model coefficients

In [ ]:
coef=pd.Series(model.named_steps["elasticnet"].coef_,index=X.columns)
display(coef.to_frame("Scaled Coefficient").sort_values("Scaled Coefficient"))

## 7. Practice
1. Change alpha from 0.1 to 0.01.
2. Try l1_ratio=0.2 and 0.8.
3. Find the lowest RMSE in the model comparison.
4. Check whether any coefficients are close to zero.
5. Explain Ridge, Lasso, and Elastic Net in your own words.

**Key points:** Elastic Net combines L1 and L2 penalties. Alpha controls regularization strength; l1_ratio controls the mix. Scaling is important.

Next: Review supervised learning algorithms and practise choosing a model.